# 🍛 South Asian Culinary RAG Assistant
### MSc Advanced Computer Science — Text Mining Coursework
**University of Manchester**

---

This notebook is the **single entry point** for the entire project.

| Cell | Purpose |
|------|---------|
| 1 | Environment setup & dependency check |
| 2 | Run benchmark evaluation → `output_payload_sample.json` |
| 3 | Launch Streamlit UI (live demo) |
| 4 | Display evaluation results inline |
| 5 | Stop Streamlit server |

**Architecture:** LangGraph · FAISS · `BAAI/bge-small-en-v1.5` · `Qwen/Qwen2.5-0.5B-Instruct`

---
## Cell 1 — Environment Setup

In [ ]:
import os, sys, json

# Set working directory to this notebook's own folder so all relative paths work
HERE = os.path.dirname(os.path.abspath("__file__"))
os.chdir(HERE)
sys.path.insert(0, HERE)
print(f"Working directory: {os.getcwd()}")

# Verify all required files are present
required = [
    "assistant_core.py",
    "app.py",
    "evaluate.py",
    "benchmark_dataset.json",
    os.path.join("faiss_index", "index.faiss"),
    os.path.join("faiss_index", "index.pkl"),
]
all_ok = True
for f in required:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌ MISSING'}  {f}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("One or more required files are missing.")

# Install dependencies
print("\nInstalling / verifying packages…")
!pip install -q streamlit langchain-community langchain-huggingface \
    langgraph faiss-cpu transformers torch sentence-transformers
print("All packages ready.")

---
## Cell 2 — Run Benchmark Evaluation

Runs all 15 benchmark queries through the LangGraph pipeline and computes:
- **Recall@3** — fraction of expected dishes retrieved in top-3
- **Intent Accuracy** — predicted vs expected intent
- **Latency** — wall-clock seconds per query

Results are saved to `output_payload_sample.json`.

> ⏱️ First run takes ~60–90 s while Qwen downloads. Subsequent runs are faster.

In [ ]:
!python evaluate.py

---
## Cell 3 — Launch Streamlit UI

Launches `app.py` as a background process.

After ~5 seconds, open: **http://localhost:8501**

> Run Cell 5 to stop the server when done.

In [ ]:
import subprocess, time, sys

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.fileWatcherType", "none",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

time.sleep(5)
print("✅ Streamlit is running!")
print("   👉  Open your browser at:  http://localhost:8501")
print(f"   Process PID: {streamlit_proc.pid}")
print("\n   Run Cell 5 to stop the server.")

---
## Cell 4 — Display Evaluation Results

In [ ]:
import json
from IPython.display import display, Markdown

with open("output_payload_sample.json", "r", encoding="utf-8") as f:
    payload = json.load(f)

summary = payload["evaluation_summary"]
results = payload["per_query_results"]

display(Markdown("### 📊 Evaluation Summary"))
display(Markdown(
    f"| Metric | Value |\n"
    f"|--------|-------|\n"
    f"| Total Queries | {summary['total_queries']} |\n"
    f"| Mean Recall@3 | {summary.get('mean_recall@3', 'N/A')} |\n"
    f"| Intent Accuracy | {summary.get('intent_accuracy', 'N/A')} |\n"
    f"| Mean Latency (s) | {summary['mean_latency_seconds']} |\n"
    f"| Min Latency (s) | {summary['min_latency_seconds']} |\n"
    f"| Max Latency (s) | {summary['max_latency_seconds']} |\n"
))

display(Markdown("### 🔍 Per-Query Results"))
header = "| ID | Query | Expected Intent | Predicted Intent | ✓ | Recall@3 | Latency (s) |\n"
header += "|----|----|----|----|----|----|-----|\n"
rows = ""
for r in results:
    tick   = "✅" if r["intent_correct"] else "❌"
    recall = r.get("recall@3", "—")
    rows += (
        f"| {r['id']} "
        f"| {r['query'][:45]} "
        f"| {r['expected_intent']} "
        f"| {r['predicted_intent']} "
        f"| {tick} "
        f"| {recall} "
        f"| {r['latency_seconds']} |\n"
    )
display(Markdown(header + rows))

---
## Cell 5 — Stop Streamlit Server

In [ ]:
try:
    streamlit_proc.terminate()
    print(f"✅ Streamlit server (PID {streamlit_proc.pid}) stopped.")
except NameError:
    print("No active Streamlit process found in this session.")